# # WallStreetBets Sentiment Analysis — 2026 Refresh
#
# This notebook now runs fully from public data sources and the shared project pipeline.
# It does **not** require Alpaca credentials, Pushshift, or a local `.env` file.
#
# What it does:
# - refreshes the latest top 10 `r/wallstreetbets` tickers,
# - loads current Yahoo Finance prices,
# - merges price features with daily WSB mention/sentiment features,
# - trains a lightweight CPU-friendly LSTM for one selected latest ticker,
# - saves fresh notebook outputs under `outputs/latest_wsb_analysis/`.


# ## Step 1 — Setup


In [1]:
from pathlib import Path
import json
import sys
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch import nn

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

if not (ROOT / "src").exists():
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.wsb_latest.pipeline import (
    compute_price_features,
    ensure_nltk_resources,
    fetch_price_history,
    run_pipeline,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)
torch.manual_seed(42)
ensure_nltk_resources()

device = torch.device("cpu")
print(f"Project root: {ROOT}")
print(f"Torch device: {device}")


Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets
Torch device: cpu


# ## Step 2 — Notebook configuration
#
# Set `PREFERRED_TICKER` to one of the current top-10 symbols if you want to force a specific stock.
# Leave it as `None` to use the latest highest-ranked ticker automatically.


In [2]:
import os
TOP_N = 10
PER_FEED = 100
PRICE_PERIOD = "1y"
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
PREFERRED_TICKER = None
WINDOW_SIZE = 10
EPOCHS = 20
LEARNING_RATE = 0.01

REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"

# ## Step 3 — Refresh the latest WallStreetBets analysis artifacts


In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
mentions_path = OUTPUT_DIR / "latest_wsb_mentions.csv"
if REFRESH_WSB_DATA or not summary_path.exists() or not mentions_path.exists():
    pipeline_result = run_pipeline(
        top_n=TOP_N,
        per_feed=PER_FEED,
        price_period=PRICE_PERIOD,
        output_dir=OUTPUT_DIR,
    )
    print(json.dumps(pipeline_result, indent=2))

# ## Step 4 — Load the latest generated outputs


In [4]:
summary_df = cast(pd.DataFrame, pd.read_csv(str(OUTPUT_DIR / "top10_wsb_stocks.csv")))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
mentions_df = cast(pd.DataFrame, pd.read_csv(str(OUTPUT_DIR / "latest_wsb_mentions.csv")))
mentions_df["created_utc"] = pd.to_datetime(mentions_df["created_utc"], utc=True)

print("Latest top 10 WSB stocks")
print(summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "last_close"]].to_string(index=False))


Latest top 10 WSB stocks
ticker  mention_count  post_count  avg_sentiment  last_close
  NFLX             13           6       0.462900   74.349998
  MSFT             15           5       0.383720  401.100006
   IBM             10           5       0.166020  219.050003
  SNDK              4           4      -0.144575 1411.079956
  ASTS              6           3       0.459933   55.009998
  SPCX              5           3       0.475700  131.110001
  RKLB              4           3       0.773000   67.349998
   SPY              4           3      -0.193267  750.719971
   AFL             11           1       0.998000  123.019997
  ORCL              6           2      -0.030850  124.209999


# ## Step 5 — Select the ticker to model in this notebook


In [5]:
def select_latest_ticker(in_summary_df, preferred_ticker=None):
    if preferred_ticker:
        preferred_ticker = preferred_ticker.upper()
        if preferred_ticker in set(in_summary_df["ticker"]):
            return preferred_ticker
    return str(in_summary_df.iloc[0]["ticker"])


selected_ticker = select_latest_ticker(summary_df, PREFERRED_TICKER)
selected_summary = summary_df.loc[summary_df["ticker"].eq(selected_ticker)].iloc[0].to_dict()
print(json.dumps(selected_summary, indent=2, default=str))


{
  "ticker": "NFLX",
  "mention_count": 13,
  "post_count": 6,
  "avg_sentiment": 0.4629,
  "total_score": 1354,
  "total_comments": 771,
  "engagement": 57.71854009188868,
  "latest_mention": "2026-07-16 16:16:13+00:00",
  "rank_score": 37,
  "last_close": 74.3499984741211,
  "latest_rsi": 42.01215778459378,
  "latest_macd": -1.968933857011024,
  "return_5d": -0.0148403700605054,
  "return_21d": -0.0896289930077116,
  "rmse": 1.5733156769564147
}


# ## Step 6 — Build daily sentiment features and merge them with current price features


In [6]:
def build_daily_sentiment_features(in_mentions_df, ticker):
    ticker_df = in_mentions_df.loc[in_mentions_df["ticker"].eq(ticker)].copy()
    if ticker_df.empty:
        return pd.DataFrame(
            columns=[
                "mentions",
                "post_count",
                "mean_sentiment",
                "weighted_sentiment",
                "score_sum",
                "comment_sum",
            ]
        )

    ticker_df["date"] = ticker_df["created_utc"].dt.tz_convert(None).dt.normalize()
    ticker_df["weighted_sentiment_value"] = ticker_df["mentions"] * ticker_df["sentiment"]

    daily_df = (
        ticker_df.groupby("date")
        .agg(
            mentions=("mentions", "sum"),
            post_count=("post_id", "nunique"),
            mean_sentiment=("sentiment", "mean"),
            weighted_sentiment_value=("weighted_sentiment_value", "sum"),
            score_sum=("score", "sum"),
            comment_sum=("num_comments", "sum"),
        )
        .sort_index()
    )
    daily_df["weighted_sentiment"] = (
        daily_df["weighted_sentiment_value"] / daily_df["mentions"].replace(0, np.nan)
    ).fillna(0.0)
    daily_df.drop(columns=["weighted_sentiment_value"], inplace=True)
    return daily_df



def build_merged_ml_df(ticker, in_mentions_df, price_period="1y"):
    price_df = fetch_price_history(ticker, price_period)
    price_features_df = compute_price_features(price_df)
    price_features_df.index = pd.to_datetime(price_features_df.index).normalize()

    daily_sent_df = build_daily_sentiment_features(in_mentions_df, ticker)
    merged_df = price_features_df.join(daily_sent_df, how="left")

    fill_zero_cols = [
        "mentions",
        "post_count",
        "mean_sentiment",
        "weighted_sentiment",
        "score_sum",
        "comment_sum",
    ]
    for col in fill_zero_cols:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col].fillna(0.0)

    merged_df["next_close"] = merged_df["Close"].shift(-1)
    merged_df["next_return"] = merged_df["Close"].pct_change().shift(-1)
    merged_df["target_direction"] = (merged_df["next_return"] > 0).astype(int)
    merged_df = merged_df.dropna().copy()
    return merged_df


ml_df = build_merged_ml_df(selected_ticker, mentions_df, PRICE_PERIOD)
ml_output_path = OUTPUT_DIR / f"{selected_ticker.lower()}_notebook_ml_df.csv"
ml_df.to_csv(ml_output_path)
print(f"Saved merged ML dataframe to: {ml_output_path}")
print(ml_df.tail(10).to_string())


Saved merged ML dataframe to: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/nflx_notebook_ml_df.csv
                 Open       High        Low      Close  Adj Close    Volume  Dividends  Stock Splits  Daily Return  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return  mentions  post_count  mean_sentiment  score_sum  comment_sum  weighted_sentiment  next_close  next_return  target_direction
Date                                                                                                                                                                                                                                                                                                                                                    
2026-07-01  72.589996  74.360001  72.330002  74.190002  74.190002  43841800        0.0           0.0      0.039076            5.913396      

# ## Step 7 — Visualize the latest price and sentiment context


In [7]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(ml_df.index, ml_df["Close"], label="Close", color="#1f77b4")
axes[0].plot(ml_df.index, ml_df["30-Day Rolling SMA"], label="30-Day SMA", color="#ff7f0e")
axes[0].plot(ml_df.index, ml_df["30-Day Rolling EWMA"], label="30-Day EWMA", color="#2ca02c")
axes[0].set_title(f"{selected_ticker} latest price analysis")
axes[0].legend(loc="upper left")

axes[1].bar(ml_df.index, ml_df["mentions"], label="WSB mentions", color="#9467bd")
axes[1].set_ylabel("Mentions")
axes[1].legend(loc="upper left")

axes[2].plot(ml_df.index, ml_df["weighted_sentiment"], label="Weighted sentiment", color="#d62728")
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[2].set_ylabel("Sentiment")
axes[2].legend(loc="upper left")

plt.tight_layout()
plt.show()


# ## Step 8 — Prepare sequence data for a lightweight LSTM


In [8]:
def build_feature_list(in_df):
    candidate_cols = [
        "Close",
        "Volume",
        "mentions",
        "post_count",
        "mean_sentiment",
        "weighted_sentiment",
        "score_sum",
        "comment_sum",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    return [col for col in candidate_cols if col in in_df.columns]


feature_cols = build_feature_list(ml_df)
print("Features used for the notebook LSTM:")
print(feature_cols)


Features used for the notebook LSTM:
['Close', 'Volume', 'mentions', 'post_count', 'mean_sentiment', 'weighted_sentiment', 'score_sum', 'comment_sum', '30-Day Rolling STD', '30-Day Rolling SMA', '30-Day Rolling EWMA', 'RSI', 'MACD', '5D Return', '21D Return']


In [9]:
def make_lstm_sequences(in_df, feature_columns, target_col="next_close", window_size=10):
    model_df = in_df[feature_columns + [target_col]].dropna().copy()
    if len(model_df) <= window_size + 5:
        raise ValueError("Not enough rows to build LSTM sequences.")

    train_cutoff = max(int(len(model_df) * 0.8), window_size + 1)
    train_cutoff = min(train_cutoff, len(model_df) - 1)

    x_scaler = MinMaxScaler()
    y_scaler = MinMaxScaler()
    x_scaler.fit(model_df.iloc[:train_cutoff][feature_columns])
    y_scaler.fit(model_df.iloc[:train_cutoff][[target_col]])

    X_scaled = x_scaler.transform(model_df[feature_columns])
    y_scaled = y_scaler.transform(model_df[[target_col]])

    X_values = []
    y_values = []
    index_values = []

    for i in range(window_size, len(model_df)):
        X_values.append(X_scaled[i - window_size : i])
        y_values.append(y_scaled[i])
        index_values.append(model_df.index[i])

    X_values = np.array(X_values, dtype=np.float32)
    y_values = np.array(y_values, dtype=np.float32)

    split = max(int(len(X_values) * 0.8), 1)
    split = min(split, len(X_values) - 1)

    return (
        X_values[:split],
        X_values[split:],
        y_values[:split],
        y_values[split:],
        index_values[split:],
        y_scaler,
    )


# Attempt to create LSTM sequences with graceful error handling
X_train = None
X_test = None
y_train = None
y_test = None
test_index = None
y_scaler = None
lstm_training_failed = False

try:
    X_train, X_test, y_train, y_test, test_index, y_scaler = make_lstm_sequences(
        ml_df,
        feature_cols,
        target_col="next_close",
        window_size=WINDOW_SIZE,
    )
    print(f"Training sequences: {len(X_train)}")
    print(f"Testing sequences: {len(X_test)}")
except ValueError as e:
    lstm_training_failed = True
    print(f"⚠️  WARNING: LSTM training skipped due to insufficient data: {e}")
    print(f"Available rows after feature engineering: {len(ml_df)}")
    print(f"Required minimum: {WINDOW_SIZE + 5}")


Training sequences: 168
Testing sequences: 43


# ## Step 9 — Train a lightweight PyTorch LSTM


In [10]:
class PriceLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.output(out)



def train_lstm_model(
    X_train_values,
    X_test_values,
    y_train_values,
    y_test_values,
    out_index,
    scaler_y,
    epochs=20,
    learning_rate=0.01,
):
    model = PriceLSTM(input_size=X_train_values.shape[2]).to(device)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    X_train_tensor = torch.tensor(X_train_values, dtype=torch.float32, device=device)
    X_test_tensor = torch.tensor(X_test_values, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train_values, dtype=torch.float32, device=device)

    history = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        predictions = model(X_train_tensor)
        loss = loss_fn(predictions, y_train_tensor)
        loss.backward()
        optimizer.step()
        history.append(float(loss.item()))

    model.eval()
    with torch.no_grad():
        predicted_scaled = model(X_test_tensor).cpu().numpy()

    actual_values = scaler_y.inverse_transform(y_test_values)
    predicted_values = scaler_y.inverse_transform(predicted_scaled)
    rmse = float(np.sqrt(mean_squared_error(actual_values, predicted_values)))

    results_df = pd.DataFrame(
        {
            "Actual": actual_values.ravel(),
            "Predicted": predicted_values.ravel(),
        },
        index=list(out_index),
    )
    return model, history, rmse, results_df


model = None
training_loss_history = []
lstm_rmse = None
lstm_results_df = pd.DataFrame()

if not lstm_training_failed and X_train is not None:
    try:
        model, training_loss_history, lstm_rmse, lstm_results_df = train_lstm_model(
            X_train,
            X_test,
            y_train,
            y_test,
            test_index,
            y_scaler,
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
        )
        print(f"{selected_ticker} notebook LSTM RMSE: {lstm_rmse:.4f}")
        print(f"Final training loss: {training_loss_history[-1]:.6f}")
    except Exception as e:
        lstm_rmse = None
        lstm_results_df = pd.DataFrame()
        print(f"⚠️  WARNING: LSTM model training failed: {e}")
else:
    print("⚠️  LSTM training skipped (insufficient data from sequence creation phase)")


NFLX notebook LSTM RMSE: 8.8412
Final training loss: 0.019861


# ## Step 10 — Review the LSTM output (if available)


In [11]:
lstm_results_path = OUTPUT_DIR / f"{selected_ticker.lower()}_notebook_lstm_results.csv"
lstm_plot_path = OUTPUT_DIR / "charts" / f"{selected_ticker.lower()}_notebook_lstm.png"

if not lstm_results_df.empty:
    lstm_results_df.to_csv(lstm_results_path)

    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

    axes[0].plot(training_loss_history, color="#1f77b4")
    axes[0].set_title("Training loss")
    axes[0].set_ylabel("MSE loss")

    axes[1].plot(lstm_results_df.index, lstm_results_df["Actual"], label="Actual", color="#2ca02c")
    axes[1].plot(lstm_results_df.index, lstm_results_df["Predicted"], label="Predicted", color="#d62728")
    axes[1].set_title(f"{selected_ticker} next-close prediction")
    axes[1].legend(loc="upper left")

    plt.tight_layout()
    plt.savefig(lstm_plot_path, dpi=180)
    plt.show()

    print(f"Saved LSTM results to: {lstm_results_path}")
    print(f"Saved LSTM chart to: {lstm_plot_path}")
    print(lstm_results_df.tail(10).to_string())
else:
    print(f"⚠️  LSTM results skipped (model training did not complete)")


Saved LSTM results to: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/nflx_notebook_lstm_results.csv
Saved LSTM chart to: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/charts/nflx_notebook_lstm.png
               Actual  Predicted
2026-07-01  77.650002  83.948898
2026-07-02  76.019997  84.044075
2026-07-06  76.180000  84.720337
2026-07-07  75.589996  84.754951
2026-07-08  75.470001  84.762154
2026-07-09  73.370003  84.763321
2026-07-10  73.830002  84.665504
2026-07-13  73.529999  84.293968
2026-07-14  73.680000  84.149254
2026-07-15  74.349998  73.303116


# ## Step 11 — Final notebook summary


In [12]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    value = in_df[column_name].iloc[-1]
    if pd.isna(value):
        return None
    return caster(value)


notebook_summary = {
    "selected_ticker": selected_ticker,
    "latest_close": safe_latest_value(ml_df, "Close", float),
    "latest_rsi": safe_latest_value(ml_df, "RSI", float),
    "latest_macd": safe_latest_value(ml_df, "MACD", float),
    "latest_mentions": safe_latest_value(ml_df, "mentions", int),
    "ml_rows": int(len(ml_df)),
    "lstm_rmse": lstm_rmse,
    "lstm_trained": lstm_rmse is not None,
    "top10_output_dir": str(OUTPUT_DIR.resolve()),
}
if ml_df.empty:
    print(f"WARNING: {selected_ticker} produced no usable rows after feature engineering.")
print(json.dumps(notebook_summary, indent=2))


{
  "selected_ticker": "NFLX",
  "latest_close": 73.68000030517578,
  "latest_rsi": 39.44881095301978,
  "latest_macd": -2.1047918844357554,
  "latest_mentions": 2,
  "ml_rows": 221,
  "lstm_rmse": 8.841155085982518,
  "lstm_trained": true,
  "top10_output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
